# Data Modelling Pipeline

## Objective

This notebook transforms the cleaned Olist dataset into structured, analysis-ready data models for business intelligence and analytics.

The process integrates multiple relational tables, performs data transformation and aggregation, and constructs analytical tables suitable for reporting and visualization.

## Tasks

The notebook performs:

- Integration of multiple relational datasets.
- Data transformation and aggregation.
- Construction of fact and dimension tables.
- Preparation of analytical datasets for business intelligence.

## Output

The pipeline produces:

- **Fact Table**
  - `fact_orders`

- **Dimension Tables**
  - `dim_customers`
  - `dim_products`

These datasets support:

- Power BI dashboards
- Customer analytics
- RFM analysis

Processed datasets are exported for loading into the PostgreSQL data warehouse (Supabase).

In [39]:
from pathlib import Path
import pandas as pd

In [40]:
PROJECT_ROOT = Path("../")

CLEANED_DATA_DIR = (
    PROJECT_ROOT 
    / "data"
    / "processed"
)

WAREHOUSE_DIR = (
    PROJECT_ROOT
    /"data"
    /"warehouse"
)

WAREHOUSE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [41]:
customers = pd.read_csv(
    CLEANED_DATA_DIR /
    "customers_cleaned.csv"
)


orders = pd.read_csv(
    CLEANED_DATA_DIR /
    "orders_cleaned.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)


order_items = pd.read_csv(
    CLEANED_DATA_DIR /
    "order_items_cleaned.csv"
)


payments = pd.read_csv(
    CLEANED_DATA_DIR /
    "payments_cleaned.csv"
)


reviews = pd.read_csv(
    CLEANED_DATA_DIR /
    "reviews_cleaned.csv"
)


products = pd.read_csv(
    CLEANED_DATA_DIR /
    "products_cleaned.csv"
)


sellers = pd.read_csv(
    CLEANED_DATA_DIR /
    "sellers_cleaned.csv"
)


geolocation = pd.read_csv(
    CLEANED_DATA_DIR /
    "geolocation_cleaned.csv"
)


translation = pd.read_csv(
    CLEANED_DATA_DIR /
    "product_category_translation_cleaned.csv"
)

### Geolocation Lookup Table

In [42]:
geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
738327,99965,-28.180655,-52.034367,agua santa,RS
738328,99950,-28.072188,-52.011272,tapejara,RS
738329,99950,-28.068864,-52.012964,tapejara,RS
738330,99950,-28.068639,-52.010705,tapejara,RS


In [43]:
# Aggregate multiple zip code to single location
geo_lookup = (
    geolocation
    .groupby(
        "geolocation_zip_code_prefix"
    )
    .agg(
        {
            "geolocation_lat":"mean",
            "geolocation_lng":"mean",
            "geolocation_city":"first",
            "geolocation_state":"first"
        }
    )
    .reset_index()
)

In [44]:
geo_lookup = geo_lookup.rename(
    columns={
        "geolocation_zip_code_prefix":
            "zip_code_prefix",

        "geolocation_lat":
            "latitude",

        "geolocation_lng":
            "longitude",

        "geolocation_city":
            "city",

        "geolocation_state":
            "state"
    }
)

In [45]:
geo_lookup.head()

,zip_code_prefix,latitude,longitude,city,state
0,1001,-23.550227,-46.634039,sao paulo,SP
1,1002,-23.547657,-46.634991,sao paulo,SP
2,1003,-23.549000,-46.635582,sao paulo,SP
3,1004,-23.549829,-46.634792,sao paulo,SP
4,1005,-23.549547,-46.636406,sao paulo,SP


### Dim_Customer

In [46]:
dim_customer = customers.copy()

In [47]:
dim_customer = dim_customer[
[
"customer_id",
"customer_unique_id",
"customer_zip_code_prefix",
"customer_city",
"customer_state"
]
]

In [48]:
dim_customer = dim_customer.rename(
    columns={
        "customer_id":
            "customer_key"
    }
)

In [49]:
dim_customer = dim_customer.merge(

    geo_lookup,

    left_on=
    "customer_zip_code_prefix",

    right_on=
    "zip_code_prefix",

    how="left"

)

In [50]:
dim_customer = dim_customer.drop(
    columns=[
        "zip_code_prefix"
    ]
)

### Dim Product

In [51]:
# Add English Category Translation
products = products.merge(

    translation,

    on=
    "product_category_name",

    how="left"

)

In [52]:
dim_product = products.copy()

In [53]:
dim_product = dim_product[
[
"product_id",
"product_category_name",
"product_category_name_english",
"product_weight_g",
"product_length_cm",
"product_height_cm",
"product_width_cm"
]
]

In [54]:
dim_product = dim_product.rename(
    columns={
        "product_id":
        "product_key"
    }
)

In [55]:
dim_product[
"product_category_name_english"
] = (
dim_product[
"product_category_name_english"
]
.fillna("(unknown)")
)

### Dim Seller

In [56]:
dim_seller = sellers.copy()

In [57]:
dim_seller = dim_seller[
[
"seller_id",
"seller_zip_code_prefix",
"seller_city",
"seller_state"
]
]

In [58]:
dim_seller = dim_seller.rename(
    columns={
        "seller_id":
        "seller_key"
    }
)

In [59]:
dim_seller = dim_seller.merge(

    geo_lookup,

    left_on=
    "seller_zip_code_prefix",

    right_on=
    "zip_code_prefix",

    how="left"

)

In [60]:
dim_seller = dim_seller.drop(
    columns=[
        "zip_code_prefix"
    ]
)

### Dim Payment

In [61]:
dim_payment = (

payments[
"payment_type"
]

.drop_duplicates()

.reset_index(drop=True)

)

In [62]:
dim_payment = pd.DataFrame(
{
"payment_key":
range(
1,
len(dim_payment)+1
),

"payment_type":
dim_payment
}
)

### Dim Review

In [63]:
dim_review = reviews[
[
"review_id",
"review_score",
"review_comment_message"
]
]

In [64]:
dim_review = dim_review.rename(
    columns={
        "review_id":
        "review_key"
    }
)

### Fact Orders
- One row = one product purchased inside an order

In [65]:
fact_orders = orders.merge(

    customers[
        [
        "customer_id",
        "customer_unique_id"
        ]
    ],

    on="customer_id",

    how="left"

)

In [66]:
# Add Order Items
fact_orders = fact_orders.merge(

    order_items,

    on="order_id",

    how="left"

)

In [67]:
# Add Revenue
fact_orders[
"order_revenue"
] = (

fact_orders["price"]

+

fact_orders["freight_value"]

)

In [68]:
# Add Delivery Duration
fact_orders[
"delivery_days"
] = (

fact_orders[
"order_delivered_customer_date"
]

-

fact_orders[
"order_purchase_timestamp"
]

).dt.days

In [69]:
# Add Review Score
fact_orders = fact_orders.merge(

reviews[
[
"order_id",
"review_score"
]
],

on="order_id",

how="left"

)

In [70]:
fact_orders = fact_orders[
[
"order_id",

"customer_unique_id",

"product_id",

"seller_id",

"order_status",

"order_purchase_timestamp",

"order_approved_at",

"order_delivered_carrier_date",

"order_delivered_customer_date",

"order_estimated_delivery_date",

"order_revenue",

"delivery_days",

"review_score"

]
]

### Data Warehouse Validation

In [71]:
warehouse_summary = pd.DataFrame({

"table":[

"dim_customer",

"dim_product",

"dim_seller",

"dim_payment",

"dim_review",

"fact_orders"

],

"rows":[

len(dim_customer),

len(dim_product),

len(dim_seller),

len(dim_payment),

len(dim_review),

len(fact_orders)

]

})


warehouse_summary

,table,rows
0,dim_customer,99441
1,dim_product,32951
2,dim_seller,3095
3,dim_payment,5
4,dim_review,99224
5,fact_orders,114092


#### Check Customer Relationship

In [72]:
fact_orders[
"customer_unique_id"
].isna().sum()

np.int64(0)

#### Check Products

In [73]:
missing_products = (

~fact_orders["product_id"]
.isin(
dim_product["product_key"]
)

).sum()


print(
missing_products
)

778


#### Check Sellers

In [74]:
missing_sellers = (

~fact_orders["seller_id"]
.isin(
dim_seller["seller_key"]
)

).sum()


print(
missing_sellers
)

778


### Export Data Warehouse Tables

In [75]:
dim_customer.to_csv(

WAREHOUSE_DIR /
"dim_customer.csv",

index=False

)


dim_product.to_csv(

WAREHOUSE_DIR /
"dim_product.csv",

index=False

)


dim_seller.to_csv(

WAREHOUSE_DIR /
"dim_seller.csv",

index=False

)


dim_payment.to_csv(

WAREHOUSE_DIR /
"dim_payment.csv",

index=False

)


dim_review.to_csv(

WAREHOUSE_DIR /
"dim_review.csv",

index=False

)


fact_orders.to_csv(

WAREHOUSE_DIR /
"fact_orders.csv",

index=False

)